
# 6. Replica exchange

Windowing (tutorial 5) made each walker's job small and parallel — but it did so
by *confining* each walker to an energy band. That confinement has a failure
mode: if exploring a window properly requires briefly leaving it, the confined
walker simply cannot, and gets stuck. Plain, full-range Wang-Landau never had
this problem, because one walker roams the whole axis.

**Replica-exchange Wang-Landau (REWL)** keeps windowing's parallelism but repairs
the confinement: every so often, neighbouring windows **swap configurations**. A
configuration can then travel to a window where the troublesome move *is*
possible, make it there, and come back. This tutorial builds a model where
windowing fails, confirms plain WL handles it, and shows exchange restoring it.
The maths is in :doc:`/theory/08-replica-exchange`.


## A model that traps a confined walker

Two "conformations" (basins) share a single **gateway** state at order
parameter ``q = 0`` and have *different* densities of states,
``d0(q) = C(M0, q)`` and ``d1(q) = C(M1, q)``. A move flips one bit, changing
``q`` by ``±1`` *within* a basin; the basin can change **only** at the gateway.
So to switch basins a walker must first walk all the way down to ``q = 0``.



In [ ]:
import sys
from math import comb

try:
    from pathlib import Path

    sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "examples"))
except NameError:
    pass  # sphinx-gallery exec context: __file__ undefined, sys.path is already set

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import two_basin  # noqa: E402

from flatwalk import (  # noqa: E402
    Bin1D,
    RewlDriver,
    WLConfig,
    WLDriver,
    join_g,
    make_windows,
)

M0, M1 = 8, 12
scheme = Bin1D(-0.5, M0 + 0.5, M0 + 1)
windows = make_windows(scheme, n_windows=4, overlap=3)
log_g_exact = two_basin.exact_log_g(M0, M1)
cb = two_basin.make_two_basin_callbacks(M0, M1)

Its anatomy. The two basins have different shapes and only touch at the gateway.
Any window above the gateway holds *both* basins' states, but a walker confined
there can never cross between them — to switch it would have to exit the window.



In [ ]:
qs = np.arange(M0 + 1)
d0 = np.array([comb(M0, q) for q in qs])
d1 = np.array([comb(M1, q) for q in qs])

fig, ax = plt.subplots(figsize=(6.5, 4))
for lo, hi in windows:
    ax.axvspan(lo, hi, color="C7", alpha=0.10)
ax.semilogy(qs, d0, "o-", color="C0", label="basin 0:  C(8, q)")
ax.semilogy(qs, d1, "s-", color="C1", label="basin 1:  C(12, q)")
ax.semilogy(qs, np.exp(log_g_exact), "k--", lw=1.2, label="total g(q)")
ax.annotate(
    "gateway\n(q=0)",
    xy=(0, 1),
    xytext=(1.2, 3),
    arrowprops=dict(arrowstyle="->", lw=1),
    fontsize=9,
)
ax.set_xlabel("q")
ax.set_ylabel("number of states")
ax.set_title("Two-basin model: different shapes, one gateway")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
plt.show()

## Helpers

``max ε`` against the exact ``g`` for a recovered ``log g`` over visited bins.



In [ ]:
def eps_vs_exact(g, visited):
    ne = np.exp(log_g_exact[visited])
    nw = np.exp(g[visited] - g[visited].max())
    nw *= ne.sum() / nw.sum()
    return float((np.abs(nw - ne) / ne).max())


def shifted(g, visited):
    return scheme.centers[visited], g[visited] - g[visited].min()

## Plain WL handles it — the system is fine

One full-range walker, no windows. It can walk down to the gateway, cross to the
other basin, and back, so it samples both and recovers ``g``.



In [ ]:
rng = np.random.default_rng(0)
init1 = two_basin.initial_states_for_windows(M0, M1, [(0, M0)], rng)
plain = WLDriver(
    WLConfig(bin_scheme=scheme, beta=0.0, n_check=1_000, ln_f_final=1e-5)
).run_batched(
    initial_state=init1,
    energy_fn=cb["energy_fn"],
    order_parameter_fn=cb["order_parameter_fn"],
    propose_move_fn=cb["propose_move_fn"],
    n_walkers=1,
    rng=rng,
)
eps_plain = eps_vs_exact(plain.g, plain.visited)
print(f"plain WL (one full-range walker): max ε = {eps_plain:.2f}")
assert eps_plain < 0.35

## Windowing breaks it; exchange repairs it

The same model, now windowed. Without exchange each window is trapped in the
basin it started in (it can never reach the gateway) and reports the wrong
``g``. Turn exchange on and configurations ferry through the gateway window to
every other window — recovering ``g`` while keeping the windows parallel.



In [ ]:
def run_windowed(n_exchange, seed):
    rng = np.random.default_rng(seed)
    init = two_basin.initial_states_for_windows(M0, M1, windows, rng)
    r = RewlDriver(
        WLConfig(bin_scheme=scheme, beta=0.0, n_check=1_000, ln_f_final=1e-5), windows
    ).run(
        initial_state=init,
        energy_fn=cb["energy_fn"],
        order_parameter_fn=cb["order_parameter_fn"],
        propose_move_fn=cb["propose_move_fn"],
        n_exchange=n_exchange,
        rng=rng,
        max_trials=5_000_000,
    )
    return join_g(r.g_windows, r.visited_windows)


g_off, vis_off = run_windowed(n_exchange=None, seed=0)
g_on, vis_on = run_windowed(n_exchange=5, seed=0)
eps_off = eps_vs_exact(g_off, vis_off)
eps_on = eps_vs_exact(g_on, vis_on)
print(f"windowed, no exchange: max ε = {eps_off:.2f}")
print(f"REWL (exchange on):    max ε = {eps_on:.2f}")
assert eps_off > 0.5 and eps_on < 0.35

All three against the exact curve: plain WL and REWL land on it; windowing alone
does not.



In [ ]:
q_ex = scheme.centers
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(q_ex, log_g_exact - log_g_exact.min(), "k--", lw=1.2, label="exact")
ax.plot(
    *shifted(plain.g, plain.visited),
    "^-",
    color="C2",
    label=f"plain WL, no windows (ε={eps_plain:.2f})",
)
ax.plot(
    *shifted(g_off, vis_off),
    "o-",
    color="C3",
    label=f"windowed, no exchange (ε={eps_off:.2f})",
)
ax.plot(*shifted(g_on, vis_on), "s-", color="C0", label=f"REWL (ε={eps_on:.2f})")
ax.set_xlabel("q")
ax.set_ylabel("log g(q)   (shifted to min = 0)")
ax.set_title("Only windowing-without-exchange fails")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

The lesson is precise: the trouble was never the *system* (plain WL was fine),
it was the *confinement* windowing imposed — and exchange gives it back the
reach a single walker had, without giving up the parallelism. This is exactly
parallel tempering's mechanism, and it is what makes REWL the method of choice
for rugged, first-order landscapes where even a single walker would struggle.

That completes the journey from a single fixed-temperature Monte Carlo run to a
robust, parallel estimate of the whole density of states. The
:doc:`final tutorial <plot_7_thermodynamics>` turns a converged ``g(E)`` into
the full thermodynamics — free energy, entropy, energy, and heat capacity
across temperature.

